# Fibroblast-only scProto vs. SEACells: scoped affinity fix

Follow-up to `plan1_niche_recovery_eval.ipynb` and `affinity_smoothing_diagnostic.ipynb`.
Diagnosis so far: scProto's metacells have *higher* niche purity than SEACells (0.78 vs 0.51)
but far worse gene-level niche recovery, traced to extreme metacell size-skew (median 1-4 real
cells per group) — consistent with the `ctx` affinity's spatial-context averaging mixing
cell types together (no cell-type filtering in `build_context`/`spatial_context_pca`).

**Fix tested here** (decided against a cell-type-composition feature — too close to leaking
CT labels into the affinity): train scProto on a **Fibroblasts-only** subset. The spatial
context feature (`X_ctx`) is computed on the full tissue *before* subsetting (so it still
reflects each cell's true, mixed-cell-type neighbourhood — consistent with how the source
paper itself defines niches), but the affinity graph is then built only on the Fibroblast
subset, so it's same-cell-type by construction — no explicit cross-cell-type edge masking
needed.

**Radius, not a fixed k**: `X_ctx` is built with `build_context` (radius-based, unweighted
mean PCA) — matching both the NSCLC paper's own method (a fixed physical radius, not a
neighbour count) and our own appendix's `app:spatial_affinity` formula. The radius itself is
**calibrated automatically, every time this runs**
(`spatial_subsets.calibrate_radius_for_target_median_neighbours`): the NSCLC paper reports a
median of 32 cells within their 50µm 2D neighbourhood (`nscl.pdf` p.9) — since we don't know
whether our coordinate units match their micrometers, we instead find the radius that gives
a median of 32 neighbours *in this dataset's own coordinate units*, computed exactly (median
distance to each cell's 32nd-nearest spatial neighbour), not via trial-and-error search.
Logged clearly each run so you can see the resulting radius and the actual median neighbour
count it produced.

All reusable logic lives in the codebase:
- `interpretable_ssl.datasets.spatial_subsets.build_celltype_subset_with_context` — data prep.
- `fibnsc` dataset entry in `interpretable_ssl/datasets/dataset_configs.py` — new, permanent.
- `interpretable_ssl.evaluation.niche_program_recovery` — Setup A/B/C + Branch 1/2, ported
  from `plan1_niche_recovery_eval.ipynb` (which is left as-is, already validated) so this
  notebook doesn't duplicate that logic.

This notebook's cells just call those functions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Same numpy-safe install sequence as the other notebooks in this folder. No anndata pin
# -- tried pinning anndata==0.10.6 (a version already used elsewhere in this codebase,
# batch_correct_baselines.py's _fix_arrow_strings) but it's too OLD for scarches/scvi-tools
# (missing the `anndata.io` submodule) while still too NEW for SEACells (missing the
# `dtype=` kwarg SEACells.core.summarize_by_SEACell calls) -- there may be no single
# anndata version that satisfies both. The actual SEACells dtype= TypeError is now patched
# directly at its call site in metacell_metrics.py (checks the real installed anndata
# signature, so it's correct regardless of version), so no anndata pin is needed here --
# matches the unpinned install in the original working train_seacell.ipynb / seacell.ipynb.
!pip install -q scarches faiss-cpu scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
# pybanksy -- needed for AFFINITY_TYPE='bk32' (graph_generator.py's compute_banksy_embedding
# imports `from banksy.initialize_banksy import initialize_banksy` and
# `from banksy.embed_banksy import generate_banksy_matrix`, both real pybanksy modules,
# confirmed against prabhakarlab/Banksy_py's own README before adding this line -- not
# guessed). Installed before the numpy pins below so those pins have the last word on
# numpy's version, same ordering already used for scarches/SEACells.
!pip install -q pybanksy
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy statsmodels
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"
# python-igraph/leidenalg -- only needed for graph_collapse_diagnostics's community-size
# check (Step 1.5 below); everything else in this notebook works without them.
!pip install -q python-igraph leidenalg

In [ ]:
# IMPORTANT: restart the runtime now (Runtime -> Restart session) before running the
# cells below -- see plan1_niche_recovery_eval.ipynb for why.

In [ ]:
!pip install -q pybanksy


In [ ]:
_checks = {
    "numpy": "numpy", "scipy": "scipy", "anndata": "anndata", "scanpy": "scanpy",
    "scarches": "scarches", "scvi-tools": "scvi", "seacells": "SEACells",
    "palantir": "palantir", "scib-metrics": "scib_metrics",
    "statsmodels": "statsmodels", "faiss": "faiss", "pybanksy": "banksy",
}
_failed = []
for pkg, mod in _checks.items():
    try:
        __import__(mod)
    except ImportError as e:
        _failed.append((pkg, str(e)))
if _failed:
    print("FAILED imports (fix before continuing, then restart runtime again):")
    for pkg, err in _failed:
        print(f"  {pkg}: {err}")
else:
    print("All packages import cleanly.")

All packages import cleanly.


In [ ]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [ ]:
import os
import shutil
import pandas as pd

from interpretable_ssl.configs.paths import CODE_DIR, get_dataset_model_dir, get_seacell_model_dir
from interpretable_ssl.configs.defaults import get_defaults
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.datasets.spatial_subsets import (
    build_celltype_subset_with_context, subset_matches, DEFAULT_TARGET_MEDIAN_NEIGHBOURS,
)
from interpretable_ssl.augmenters.graph_generator import (
    load_or_build_affinity, graph_collapse_diagnostics, leiden_resolution_sweep,
)
from interpretable_ssl.evaluation.batch_correct_baselines import (
    run_seacells_on_latent, save_soft_assignments, _topk_sparsify_rows,
)
from interpretable_ssl.evaluation.paper_figures import _resolve_run_dir, _list_subdirs
from interpretable_ssl.evaluation.niche_program_recovery import (
    compute_ground_truth, save_ground_truth, load_ground_truth,
    majority_label_metacells, load_cell_assignments, per_pair_diagnostics,
    build_pseudobulk, branch1_recovery, branch2_recovery, macro_average,
    load_soft_assignments, soft_label_metacells, build_soft_pseudobulk,
    size_concentration_summary,
)

# s28nsc, not ss28nsc -- s28nsc is the full/un-subsampled section-28 dataset (ss28nsc is a
# subsample of it, used for the whole-tissue plan1 run). Since we're now scoped to one cell
# type, starting from the larger source gives more Fibroblasts to build metacells from.
SRC_DS_ID = 's28nsc'
DS_ID = 'fibnsc'
CT_VALUE = 'Fibroblasts'
CT_KEY = 'celltypes'
NICHE_KEY = 'niches_2D'
TARGET_MEDIAN_NEIGHBOURS = DEFAULT_TARGET_MEDIAN_NEIGHBOURS  # 32, from nscl.pdf p.9

# The one knob to turn for a different experiment: change this, then re-run every cell
# below top to bottom. Everything downstream (graph diagnostics, scProto training,
# SEACells-on-this-graph, and the final report/run_dirs resolution) reads AFFINITY_TYPE
# instead of a hardcoded string, so switching it and re-running is enough -- no other
# cell needs editing. Must match a real `affinity_type` value generate_affinity()
# understands (e.g. 'bk32', 'bk08', 'ctxg', 'ctx', 'arbf', 'covet', ...).
#
# 'bk32': exact original pybanksy package (own expression + neighbour-mean expression,
# lambda=0.5) with num_neighbours=32, then arbf (this file's rbf_optimized) on top of the
# resulting embedding (graph_generator.py's 'bk32' branch).
#
# 'bk08': same pybanksy package, but lambda=0.8 (the paper's own domain-segmentation
# default, vs. 0.5 for 'bk32') and num_neighbours=32, with the kernel built by the REAL
# SEACells package's own SEACellGraph.rbf() (SEACells/build_graph.py) instead of this
# file's rbf_optimized reimplementation -- directly comparable to the actual package's
# own graph construction, not just our version of it (graph_generator.py's 'bk08' branch).
#
# Both need `pybanksy` installed (see the install cell above) -- import fails otherwise
# with a clear pip-install message.
AFFINITY_TYPE = 'bk08'

# affinity types that need a BANKSY embedding precomputed on the FULL tissue, before
# cell-type filtering (Step 1) -- same reasoning as X_ctx: BANKSY's neighbour-mean-
# expression component needs each cell's real, mixed-cell-type spatial neighbours, not
# just whatever same-type neighbours are left after subsetting to Fibroblasts. Each
# entry is (obsm_key, lambda_param, num_neighbours), matching generate_affinity()'s own
# per-affinity-type BANKSY calls in graph_generator.py exactly (obsm_key/lambda/k here
# MUST match the values graph_generator.py's 'bk32'/'bk08' branches use, or Step 1 will
# precompute an embedding under the right key but wrong parameters, and Step 2's
# generate_affinity call will silently reuse it without recomputing, since its own
# `if 'X_bk08' not in ad.obsm` check only looks at key presence, not parameters).
BANKSY_CONFIGS_BY_AFFINITY_TYPE = {
    'bk32': [('X_bk32', 0.5, 32)],
    'bk08': [('X_bk08', 0.8, 32)],
}
BANKSY_CONFIGS = BANKSY_CONFIGS_BY_AFFINITY_TYPE.get(AFFINITY_TYPE, [])

# scProto's max UMAP-training epochs (early stopping via `patience` still applies, so this
# is an upper bound, not a fixed count). 200 is the paper's own setting; lower it for a
# faster exploratory run on a new AFFINITY_TYPE before committing to a full-length one.
TRAIN_EPOCHS = 20

# How many prototypes to actually request -- for scProto (Step 2/3c/3d), SEACells-on-graph
# (Step 3b), AND SEACells-PCA-native (Step 3): all of them read this now, so every method
# in the final report is trained at the SAME K, a genuine apples-to-apples comparison.
# From Step 1.5's Leiden resolution sweep on AFFINITY_TYPE ('bk08'): resolution~1.5-2.0 gives
# 19-24 clean communities (zero tiny/noise ones even out past resolution=5.0/57 communities)
# -- 32 sits in that range, comfortably below the 204 (K~=N/75 convention) that collapsed to
# ~8 effectively-used prototypes. Every method's save directory now encodes K directly
# (scProto: `NP{n}` in the folder name, already true before; SEACells: a new `_K{n}` suffix
# -- see get_seacell_model_dir), so changing this is always safe and never silently
# overwrites/reuses a differently-sized run of any method.
NUM_PROTOTYPES = 32

# Settled on 'ema' only (the paper's actual setting, LAMBDA_PROTO_UMAP_PRECON's default)
# -- the 'ema' vs. 'nk' anti-collapse comparison already ran (see Step 3d/Summary), no
# need to keep training/carrying 'nk' through every downstream cell. Left as a list (not
# a bare string) so every cell that loops over PROTO_USAGE_MODES keeps working unchanged
# -- reintroduce 'nk' here later if the comparison needs revisiting.
PROTO_USAGE_MODES = ['ema']


def resolve_scproto_run_dir(ds_id, affinity_type, proto_usage_mode, num_prototypes):
    """Match against the RAW folder name, NOT extract_model_key(e)'s output --
    extract_model_key deliberately strips `_NP\\d+` (num_prototypes) as
    "dataset-specific info" (see result_tables.py:_DATASET_SPECIFIC_PATTERNS), so a
    check for 'NP{n}_' against its output can never succeed regardless of what K was
    actually used.

    Second wrinkle: generate_model_name (model_name.py) only emits a `pum-{mode}_`
    token when proto_usage_mode DIFFERS from the flat global default
    (get_defaults()['proto_usage_mode'], currently 'nk') -- a run trained with the
    default value has NO `pum-` substring in its folder name at all (confirmed: the
    real 'ema' run's folder has 'pum-ema_'; the real 'nk' run's folder has no 'pum-'
    token whatsoever). Requiring 'pum-nk_' literally can therefore never match a
    default-mode run. Derived from get_defaults() rather than hardcoded, so this keeps
    working if the global default ever changes.
    """
    base_dir = get_dataset_model_dir(ds_id)
    if not os.path.isdir(base_dir):
        return None
    default_pum = get_defaults().get('proto_usage_mode')
    need = [f'NP{num_prototypes}_', f'aff-{affinity_type}_']
    pum_token = f'pum-{proto_usage_mode}_'

    def pum_ok(e):
        if proto_usage_mode == default_pum:
            return pum_token in e or 'pum-' not in e
        return pum_token in e

    matched = [e for e in _list_subdirs(base_dir) if all(kw in e for kw in need) and pum_ok(e)]
    if not matched:
        return None
    if len(matched) > 1:
        print(f"  [resolve_scproto_run_dir] {need + [pum_token]} matched {len(matched)} runs -- using '{matched[-1]}'")
    return os.path.join(base_dir, matched[-1])


pd.set_option('display.max_columns', 50)

In [ ]:
print(f"AFFINITY_TYPE={AFFINITY_TYPE!r}  NUM_PROTOTYPES={NUM_PROTOTYPES}  "
      f"PROTO_USAGE_MODES={PROTO_USAGE_MODES}  TRAIN_EPOCHS={TRAIN_EPOCHS}")
print("^ if this doesn't match what you just edited above, this cell wasn't re-run since "
      "the edit -- re-run it (and everything below) before trusting any result.")

AFFINITY_TYPE='bk08'  NUM_PROTOTYPES=32  PROTO_USAGE_MODES=['ema']  TRAIN_EPOCHS=20
^ if this doesn't match what you just edited above, this cell wasn't re-run since the edit -- re-run it (and everything below) before trusting any result.


## Step 1 — data prep: Fibroblast subset with pre-filtering, radius-calibrated spatial context

Registered as `fibnsc` in `dataset_configs.py` (permanent). Checks the existing file's
recorded provenance (`src_ds_id`/`ct_key`/`ct_value`/`target_median_neighbours`, written into
`.uns` by `build_celltype_subset_with_context`) before deciding to skip — not just "does a
file exist at this path." (This caught a real bug: an earlier `fibnsc.h5ad` built from
`ss28nsc`, 7592 Fibroblasts, was silently kept after the source was switched to `s28nsc`,
15309 Fibroblasts, because the old check only tested path existence.)

If the recorded provenance doesn't match what we're asking for now, the stale file — and
everything derived from it (old `fibnsc` model runs, cached ground truth) — is deleted before
rebuilding, since a mismatched base dataset makes all of that invalid too, not just the file
itself.

**BANKSY, same reasoning as X_ctx**: when `AFFINITY_TYPE` needs a BANKSY embedding
(`BANKSY_CONFIGS` non-empty — currently `bk32`), it's computed here too, on the full
`s28nsc` tissue *before* subsetting to Fibroblasts — not later in Step 1.5. BANKSY's
neighbour-mean-expression term needs each cell's real, mixed-cell-type spatial neighbours;
computing it after filtering would silently redefine "neighbour" as "nearby Fibroblast",
same failure mode X_ctx was already fixed for. `generate_affinity`'s own `'bk32'` branch
already skips recomputation when `X_bk32` is already in `obsm` (matching `_ensure_X_ctx`'s
pattern), so once it's precomputed here, Step 1.5 and Step 2 both pick it up for free.

This also means two *different* kinds of "doesn't match yet": a genuinely different
src/cell-type/target (truly stale — deletes and rebuilds everything, as before) vs. the same
Fibroblast subset just missing a BANKSY embedding a newly-chosen `AFFINITY_TYPE` needs (the
cell set itself hasn't changed, so existing model runs for *other* affinity types are left
alone — only the `.h5ad` is rebuilt, to add the embedding).

In [ ]:
fib_path = DATASETS[DS_ID]['path']

base_ok = subset_matches(fib_path, SRC_DS_ID, CT_KEY, CT_VALUE, TARGET_MEDIAN_NEIGHBOURS)
full_ok = base_ok and subset_matches(
    fib_path, SRC_DS_ID, CT_KEY, CT_VALUE, TARGET_MEDIAN_NEIGHBOURS,
    banksy_configs=BANKSY_CONFIGS,
)

if full_ok:
    print(f'{fib_path} already matches src={SRC_DS_ID!r}, ct={CT_VALUE!r}, '
          f'target_median_neighbours={TARGET_MEDIAN_NEIGHBOURS}, banksy_configs={BANKSY_CONFIGS} '
          f'-- skipping regeneration')
else:
    if not base_ok and os.path.exists(fib_path):
        print(f'{fib_path} exists but does not match the requested src/cell-type/target -- '
              f'removing it and everything derived from it (any fibnsc model runs, cached '
              f'ground truth)')
        os.remove(fib_path)

        model_dir = get_dataset_model_dir(DS_ID)
        if os.path.isdir(model_dir):
            print(f'removing stale model runs at {model_dir}')
            shutil.rmtree(model_dir)

        gt_path = os.path.join(CODE_DIR, 'files', f'celltype_niches_full_{DS_ID}.csv')
        if os.path.exists(gt_path):
            print(f'removing stale cached ground truth at {gt_path}')
            os.remove(gt_path)
    elif base_ok:
        print(f'{fib_path} matches src/cell-type/target, but is missing a BANKSY embedding '
              f'AFFINITY_TYPE={AFFINITY_TYPE!r} needs (banksy_configs={BANKSY_CONFIGS}) -- '
              f'rebuilding the file to add it. NOT touching existing model runs for other '
              f'affinity types -- the Fibroblast cell set itself is unchanged, only a new '
              f'obsm embedding is being added.')

    build_celltype_subset_with_context(
        SRC_DS_ID, ct_key=CT_KEY, ct_value=CT_VALUE,
        target_median_neighbours=TARGET_MEDIAN_NEIGHBOURS, out_path=fib_path,
        banksy_configs=BANKSY_CONFIGS,
    )

print(DATASETS[DS_ID])

/content/drive/MyDrive/data/spatial/fibnsc.h5ad already matches src='s28nsc', ct='Fibroblasts', target_median_neighbours=32, banksy_configs=[('X_bk08', 0.8, 32)] -- skipping regeneration
{'path': PosixPath('/content/drive/MyDrive/data/spatial/fibnsc.h5ad'), 'label_key': 'celltypes', 'niche_key': 'niches_2D', 'label_encoder_path': '/content/drive/MyDrive/codes/interpretable-prototype/data/NSCLC_3D_section_28.pkl', 'num_prototypes': 204, 'batch_size': 512, 'ft_epochs': 0}


## Step 1.5 — graph diagnostics: predict collapse risk before training

Before spending a full training run to find out, `graph_collapse_diagnostics`
(`graph_generator.py`) checks an affinity graph directly for the failure mode already seen
in earlier results (one 2702-cell metacell, 93% of the rest under 10 cells):
- **`effk` vs. raw degree** (mean/median/std/min/max for both — same stats
  `affinity_report()` in `trainers/scproto.py` already computes and logs for whatever graph
  a real training run trains on, reproduced here so a candidate graph can be checked without
  training at all). `effk_i = 1/sum(p_ij^2)` is the effective number of neighbours a cell's
  edge weights actually spread mass over. `effk/degree` near 1.0 means the weights are close
  to flat/uniform (no local structure to separate cells on); near 0 means sharp/discriminative.
  The **spread** (std/min/max) matters as much as the mean here — a graph that's mostly
  well-behaved but has a dense subset of near-uniform-weight cells is exactly the
  few-huge-many-tiny shape, and mean/median alone can hide that.
- **Predicted usable prototype count**: `n_cells / effk_mean`, compared against `NUM_PROTOTYPES`.
  scProto's community-preserving signal (the UMAP loss on graph edges, and `nassoc` when
  `lambda_nassoc>0`) pulls each cell's effective neighbourhood toward a shared prototype, so
  if `K` is far above this predicted count, most of the excess prototypes have no
  distinguishable neighbourhood to grab onto — the "many tiny metacells" half of the failure
  mode, independent of the "few huge" half.
- **Leiden communities on the raw graph, at one fixed resolution** (no training): if one
  community already swallows a large fraction of all cells before any training happens,
  that's a strong predictor the trained model will collapse the same way.

**New: `leiden_resolution_sweep`** — the single-resolution Leiden check above only tells you
whether resolution=1.0 collapses; it doesn't tell you what a *sensible* `NUM_PROTOTYPES` would
be. This sweeps several resolutions and reports the community count, median community size,
and how many communities are tiny (< 5 cells) at each — a direct, training-free, data-driven
alternative to the fixed K ~= N/75 convention. Printed alongside the actual number of true
niches in the ground truth (`adata.obs[NICHE_KEY].nunique()`, excluding `'Excluded'`) as a
sanity floor: a resolution giving noticeably *fewer* communities than true niches means the
graph itself can't even separate the niches you're trying to recover — not something more
prototypes or better training can fix.

Compares `AFFINITY_TYPE` (set at the top of the notebook — what Step 2 actually trains on)
against `arbf` (plain PCA-based adaptive RBF, a reference known to behave well elsewhere in
this codebase — see `affinity_smoothing_diagnostic.ipynb`) on this same Fibroblast-only data,
so a flatter/more collapse-prone topology should show up here directly, without waiting on
training. If `AFFINITY_TYPE` is itself `'arbf'` the comparison is skipped (nothing to compare
against itself).

Also loads `adata` here (once) and reuses it in every cell below — Step 2/3/3b/4 no longer
each reload the same `fibnsc.h5ad` independently.

**Load-if-exists**: `load_or_build_affinity` checks for a cached `.pkl` first, using the
exact same filename convention `adata_augmenter.py`'s training pipeline already writes
(`affinity_{ds}{n_cells}_ncomp50_kneighbors50_{affinity_type}.pkl` under `./graphs`) — so if
Step 2's training run already built this graph, this reuses it instead of recomputing (the
`ctx`/`ctxg` kernel build alone took ~90s in earlier runs). Only builds fresh (and saves for
next time) if no matching cache is found. The built/loaded matrices are kept in `graph_mats`
so Step 3b (SEACells on this same graph) doesn't have to rebuild `AFFINITY_TYPE` either.

In [ ]:
import scanpy as sc

adata = sc.read_h5ad(DATASETS[DS_ID]['path'])
if 'X_pca' not in adata.obsm:
    sc.tl.pca(adata, n_comps=50)

graph_mats = {}
for aff_type in dict.fromkeys([AFFINITY_TYPE, 'arbf']):  # dedupes, keeps order
    graph_mats[aff_type] = load_or_build_affinity(adata, DS_ID, aff_type, k=50, bk=None)

[load_or_build_affinity] found cached graph at ./graphs/affinity_fibnsc15309_ncomp50_kneighbors50_bk08.pkl -- loading (not recomputing)
[load_or_build_affinity] found cached graph at ./graphs/affinity_fibnsc15309_ncomp50_kneighbors50_arbf.pkl -- loading (not recomputing)


In [ ]:
true_niches = sorted(set(adata.obs[NICHE_KEY].dropna().unique()) - {'Excluded'})
print(f"True niches in ground truth (excluding 'Excluded'): {len(true_niches)} -- {true_niches}")
print(f"Current NUM_PROTOTYPES = {NUM_PROTOTYPES} (n_cells={adata.n_obs}, "
      f"i.e. {adata.n_obs / NUM_PROTOTYPES:.1f} cells/prototype on average)\n")

leiden_sweep_results = leiden_resolution_sweep(graph_mats[AFFINITY_TYPE], name=AFFINITY_TYPE)

True niches in ground truth (excluding 'Excluded'): 9 -- ['Airways', 'Alveolar spaces', 'Desmoplastic stroma', 'Macrophage islands', 'Smooth muscle structures', 'T cell aggregates', 'Tumor core', 'Tumor surface', 'Vascular stroma']
Current NUM_PROTOTYPES = 32 (n_cells=15309, i.e. 478.4 cells/prototype on average)

[bk08] Leiden resolution sweep (n=15309 cells):
resolution   n_comm   median_size    frac<5  top1_frac
       0.1        1       15309.0      0.0%     100.0%
      0.25        4        2416.5      0.0%      60.3%
       0.5        6        2399.5      0.0%      25.2%
      0.75       10        1460.0      0.0%      17.2%
       1.0       12        1299.5      0.0%      16.3%
       1.5       19         631.0      0.0%      13.0%
       2.0       24         546.0      0.0%       8.0%
       3.0       38         378.5      0.0%       5.3%
       5.0       57         252.0      0.0%       3.6%


### Supplementary: effk/degree spread diagnostics (secondary detail beyond the sweep above)

`graph_collapse_diagnostics` at a single fixed resolution — kept for the effk/degree spread
stats and the `arbf` reference comparison, which the sweep above doesn't cover. The sweep
above is the primary tool for choosing `NUM_PROTOTYPES`; this is supplementary detail.

In [ ]:
graph_diag = {}
for aff_type in graph_mats:
    graph_diag[aff_type] = graph_collapse_diagnostics(
        graph_mats[aff_type], name=aff_type, num_prototypes=NUM_PROTOTYPES,
    )
    print()

[bk08] n=15309  nnz=1123367
[bk08] raw degree:  median=65.0  mean=73.4  std=25.0  min=51  max=491
[bk08] effk:        median=58.3  mean=65.5  std=24.3  min=15.4  max=469.3
[bk08] effk/degree ratio (median): 0.892  (near 1.0 = flat/no local structure to separate on, near 0 = sharp/discriminative)
[bk08] predicted usable prototype count: n/effk_mean=234  n/degree_mean=209  vs. requested K=32
[bk08] Leiden (resolution=1.0): 12 communities, largest 5: [2494, 2095, 1804, 1671, 1607]  top-1 fraction of all cells: 16.3%

[arbf] n=15309  nnz=1232875
[arbf] raw degree:  median=66.0  mean=80.5  std=40.6  min=51  max=625
[arbf] effk:        median=62.4  mean=76.2  std=39.5  min=42.5  max=584.1
[arbf] effk/degree ratio (median): 0.943  (near 1.0 = flat/no local structure to separate on, near 0 = sharp/discriminative)
[arbf] predicted usable prototype count: n/effk_mean=201  n/degree_mean=190  vs. requested K=32
[arbf] Leiden (resolution=1.0): 7 communities, largest 5: [2634, 2441, 2428, 2289, 2252

## Step 2 — train scProto on the Fibroblast subset, both `PROTO_USAGE_MODES`

Same base config as the paper's main run (`LAMBDA_PROTO_UMAP_PRECON`; cvae_epochs=50,
batch_size=1024, min_delta=0.005, umap_steps_per_epoch=1000 — all from
`neurips_manuscript/appendix/training.tex`), except `proto_usage_mode` is overridden per
loop iteration to compare `'ema'` (the paper's default) against `'nk'` — see the config
cell's comment for why. `affinity_type=AFFINITY_TYPE`, `num_prototypes=NUM_PROTOTYPES` —
both set at the top of the notebook.

**`skip_metrics=['task2']`**: task2 includes scGraph (`interpretable_ssl/scGraph.py`), which
computes pairwise distances *between different cell-type centroids* to check whether
metacell-level structure preserves cell-level structure. With only Fibroblasts in this
dataset there are zero pairs of *different* cell types, so that computation degenerates to
`numpy.average` on weights that sum to zero — a real `ZeroDivisionError`, not a transient
issue. This is unrelated to what Step 4 actually evaluates (that reads
`cell_assignments.csv` directly, not scProto's own task2 metrics), so skipping it costs
nothing here.

**Idempotent, `resolve_scproto_run_dir(DS_ID, AFFINITY_TYPE, pum, NUM_PROTOTYPES)`**: requires
all three of `NP{NUM_PROTOTYPES}_`, `aff-{AFFINITY_TYPE}_`, and `pum-{pum}_` to appear in the
folder name — see the config cell's `resolve_scproto_run_dir` docstring for why a single
substring keyword (as used elsewhere, e.g. `_resolve_run_dir(DS_ID, f'aff-{AFFINITY_TYPE}_')`)
isn't enough once there are two scProto variants differing only in `proto_usage_mode`, which
sits far from `aff-`/`NP` in the folder name. `NUM_PROTOTYPES` is also embedded directly in
the folder name (`NP{n}`), so a different `NUM_PROTOTYPES` always produces a distinct run —
changing it can never silently reuse a stale, differently-sized run.

In [ ]:
scproto_run_dirs = {}
for pum in PROTO_USAGE_MODES:
    run_dir = resolve_scproto_run_dir(DS_ID, AFFINITY_TYPE, pum, NUM_PROTOTYPES)
    if run_dir and os.path.exists(os.path.join(run_dir, 'cell_assignments.csv')):
        print(f'scProto ({AFFINITY_TYPE}, pum={pum}, NP={NUM_PROTOTYPES}) already '
              f'trained+evaluated at {run_dir} -- skipping.')
        scproto_run_dirs[pum] = run_dir
        continue

    lambda_config = dict(LAMBDA_PROTO_UMAP_PRECON)
    lambda_config['proto_usage_mode'] = pum

    # find_metacells (aliased as run_mc_task) returns a 3-tuple (t, metrics, mc_adata) --
    # every return path in tasks.py:find_metacells does.
    t, metrics, mc_adata = run_mc_task(
        DS_ID,
        cvae_epochs=50,
        train_epochs=TRAIN_EPOCHS,
        eval_freq=5,
        patience=10,
        batch_size=1024,
        lambda_config=lambda_config,
        affinity_type=AFFINITY_TYPE,
        label_key=CT_KEY,
        niche_key=NICHE_KEY,
        num_prototypes=NUM_PROTOTYPES,
        skip_metrics=['task2'],
    )
    print(f'[pum={pum}]', metrics)

    scproto_run_dirs[pum] = resolve_scproto_run_dir(DS_ID, AFFINITY_TYPE, pum, NUM_PROTOTYPES)
    assert scproto_run_dirs[pum], (
        f'training for pum={pum} just completed (requested NUM_PROTOTYPES={NUM_PROTOTYPES}) '
        f'but no run dir matching NP{NUM_PROTOTYPES}_/aff-{AFFINITY_TYPE}_/pum-{pum}_ was '
        f'found afterward. If the model that WAS just saved shows a different NP{{n}} in its '
        f'folder name than NUM_PROTOTYPES above, NUM_PROTOTYPES most likely changed (or was '
        f'never actually re-run) in this kernel between when you last edited the config cell '
        f'and when this cell started -- rerun the config cell, confirm its printed '
        f'NUM_PROTOTYPES matches what you intend, then rerun this cell.'
    )

scproto_run_dirs

scProto (bk08, pum=ema, NP=32) already trained+evaluated at /content/drive/MyDrive/models/fibnsc/proto_umap_ds-fibn_NP32_prtInit-wayp_aff-bk08_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31 -- skipping.


{'ema': '/content/drive/MyDrive/models/fibnsc/proto_umap_ds-fibn_NP32_prtInit-wayp_aff-bk08_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31'}

## Step 3 — train SEACells (plain, PCA-native) at `NUM_PROTOTYPES`

This is SEACells run entirely its own way: its own kernel (`build_kernel_on='X_pca'`), its
own waypoint archetype seeding — no connection to `AFFINITY_TYPE` at all. Now trained at
`NUM_PROTOTYPES` like everything else, via `train_seacell(..., num_prototypes=NUM_PROTOTYPES)`
/ `eval_seacell_task1(..., num_prototypes=NUM_PROTOTYPES)` — both now accept and thread this
through to `get_seacell_model_dir`, which appends a `_K{n}` suffix whenever `num_prototypes`
is given (`None`, the default, reproduces the exact old path for every other caller in the
codebase that doesn't pass it — this doesn't change behaviour anywhere else).

`mode='eval'`, not `mode='train'`: `train_seacell`'s own logic is `if mode == "train" or not
seacell_exists: train`, so `mode='train'` forces a full retrain on every rerun of this cell --
`mode='eval'` only trains if the base SEACells object doesn't exist yet, otherwise it's a
cheap no-op ("eval mode, seacell file found").

`train_seacell` alone does **not** produce `cell_assignments.csv` / `purity_per_mc.csv` --
those are written by a separate function, `eval_seacell_task1`. (scProto's `run_mc_task` does
training + eval + save in one call; SEACells splits training and eval into two separate
functions instead.) Skips re-running eval if it's already there, matching Step 2's
idempotency.

**Exact path, not `_resolve_run_dir`**: `get_seacell_model_dir(DS_ID, num_prototypes=NUM_PROTOTYPES)`
resolves to the exact `.../fibnsc/seacell_K{NUM_PROTOTYPES}` folder — distinct both from the
plain `.../fibnsc/seacell` folder (an older, un-K-suffixed run from before this parameter
existed, left untouched) and from Step 3b's `.../fibnsc/seacell_{graph}` folders.

In [ ]:
seacell_run_dir = get_seacell_model_dir(DS_ID, num_prototypes=NUM_PROTOTYPES)
if os.path.exists(os.path.join(seacell_run_dir, 'cell_assignments.csv')):
    print(f'SEACells (PCA-native, K={NUM_PROTOTYPES}) already trained+evaluated for {DS_ID} '
          f'at {seacell_run_dir} -- skipping.')
else:
    train_seacell(DS_ID, mode='eval', num_prototypes=NUM_PROTOTYPES)
    eval_seacell_task1(DS_ID, num_prototypes=NUM_PROTOTYPES)

SEACells (PCA-native, K=32) already trained+evaluated for fibnsc at /content/drive/MyDrive/models/fibnsc/seacell_K32 -- skipping.


## Step 3b — SEACells archetypal analysis on `arbf` AND on `AFFINITY_TYPE`, same `NUM_PROTOTYPES`

Two SEACells variants, both via `run_seacells_on_latent` (`batch_correct_baselines.py`) and
both at `NUM_PROTOTYPES` — matching scProto exactly, unlike Step 3's fixed-K PCA-native
baseline:
- **SEACells (arbf)**: same `arbf` graph used as the reference in Step 1.5's diagnostics —
  the "known-good" graph. Together with SEACells (`AFFINITY_TYPE`), this isolates whether a
  gap is about the **graph** (arbf vs. `AFFINITY_TYPE`, same clusterer) — separately from
  Step 4/5's scProto-vs-SEACells comparison, which isolates the **clusterer** (same graph,
  different clusterer).
- **SEACells (`AFFINITY_TYPE`)**: same graph scProto trains on (`graph_mats[AFFINITY_TYPE]`
  from Step 1.5).

For both: SEACells' kernel comes entirely from `aff` via `add_precomputed_kernel_matrix` (not
`build_kernel_on`, which is just a folder-naming tag here — see
`compute_seacells_own_affinity`'s docstring), and even the archetype seeding is computed from
`aff` (`waypoint_archetype_indices`) rather than SEACells' own PCA-waypoint init.

Saves to `.../fibnsc/seacell_{graph}` (`cell_assignments.csv` etc., same layout Step 4 already
reads) — one folder per graph, distinct from Step 3's plain `.../fibnsc/seacell` folder.

**Load, not generate**: `skip_if_exists=True` (the default) skips the archetypal analysis
entirely if `seacell_{graph}`'s `metrics.json` already exists *and* its realized metacell
count matches `NUM_PROTOTYPES` within 5% (`_cached_seacell_run_matches_k`) — a stale run left
over from a different `NUM_PROTOTYPES` is treated as a cache miss and recomputed, not silently
reused. Task 2 (`compute_dge=False`) is best-effort and won't fail the whole cell if it
errors; Task 1 (what Step 4 actually reads) always runs.

In [ ]:
seacell_aff_run_dirs = {}
for graph_name in dict.fromkeys(['arbf', AFFINITY_TYPE]):  # dedupes if AFFINITY_TYPE=='arbf'
    result = run_seacells_on_latent(
        DS_ID, adata,
        n_seacells=NUM_PROTOTYPES,
        aff=graph_mats[graph_name],
        tag=graph_name,
        skip_if_exists=True,
    )
    seacell_aff_run_dirs[graph_name] = result['save_path']
    print(f"SEACells ({graph_name}) -> {result['save_path']}"
          f"{'  (loaded existing run)' if result.get('skipped') else '  (freshly computed)'}")

seacell_aff_run_dirs

[fibnsc] seacell_arbf already computed -- skipping (metrics.json found)
[fibnsc] canonical ARBF-on-PCA affinity graph loaded from ./graphs/affinity_fibnsc15309_ncomp50_kneighbors50_arbf.pkl (nnz=1232875) -- caching for reuse across all methods for this dataset.
[/content/drive/MyDrive/models/fibnsc/seacell_arbf] modularity recomputed against canonical graph: mean_modularity_batch=None +/- None (was None), K_target=32
SEACells (arbf) -> /content/drive/MyDrive/models/fibnsc/seacell_arbf  (loaded existing run)
[fibnsc] seacell_bk08 already computed -- skipping (metrics.json found)
[/content/drive/MyDrive/models/fibnsc/seacell_bk08] modularity recomputed against canonical graph: mean_modularity_batch=None +/- None (was None), K_target=32
SEACells (bk08) -> /content/drive/MyDrive/models/fibnsc/seacell_bk08  (loaded existing run)


{'arbf': '/content/drive/MyDrive/models/fibnsc/seacell_arbf',
 'bk08': '/content/drive/MyDrive/models/fibnsc/seacell_bk08'}

## Step 3c — save scProto's soft assignment matrix, both `PROTO_USAGE_MODES` (loads the trained models, does NOT retrain)

SEACells (Step 3b, via `run_seacells_on_latent`) already saves a real `(n_cells x n_protos)`
soft assignment matrix. scProto's own `run_mc_task`/`save_umap_data` only ever wrote **hard**
argmax assignments to `cell_assignments.csv` — no soft matrix existed for it at all, so a
soft-label-based DGE comparison wasn't possible. This cell adds it, for both `pum` variants.

**No retraining**: `run_mc_task(..., load_umap=True, skip_eval=True)` only calls
`t.load_umap_checkpoint()` (restores the already-trained model weights + calibrated
epsilon/temperature from Step 2's own saved checkpoint) and returns immediately — the
`pretrain_encoder()`/`train_umap_edges()` calls that do actual training are inside the
`if not load_umap:` branch in `tasks.py:find_metacells`, never reached here. Same kwargs as
Step 2's own training call (including `proto_usage_mode=pum`), so `get_trainer()`
reconstructs the identical model name/path and loads the exact checkpoint Step 2 produced.

**Same formula scProto already uses internally**: `S = softmax(scores / epsilon)`, exactly
what `eval_metacell_quality(soft_metrics=True)` computes (`trainers/scproto.py`) — reproduced
directly here instead of calling that method, which also re-saves `clusters.npz`/purity CSVs
as a side effect we don't need. Sparsified to each cell's top-20 entries via
`_topk_sparsify_rows` and saved via `save_soft_assignments` — the exact same helpers and
top-k cutoff `run_seacells_on_latent` already uses for SEACells' soft matrix, so all are
comparable on equal footing.

**Load-if-exists**: skipped entirely, per `pum`, if `soft_assignments.npz` is already there.

In [ ]:
import torch
import torch.nn.functional as F

for pum in PROTO_USAGE_MODES:
    run_dir = scproto_run_dirs[pum]
    if os.path.exists(os.path.join(run_dir, 'soft_assignments.npz')):
        print(f'scProto ({AFFINITY_TYPE}, pum={pum}) soft assignments already saved at '
              f'{run_dir} -- skipping.')
        continue

    lambda_config = dict(LAMBDA_PROTO_UMAP_PRECON)
    lambda_config['proto_usage_mode'] = pum

    # load_umap=True, skip_eval=True -- loads the finished checkpoint, does NOT retrain
    # (see markdown above).
    t, _, _ = run_mc_task(
        DS_ID,
        cvae_epochs=50,
        train_epochs=TRAIN_EPOCHS,
        eval_freq=5,
        patience=10,
        batch_size=1024,
        lambda_config=lambda_config,
        affinity_type=AFFINITY_TYPE,
        label_key=CT_KEY,
        niche_key=NICHE_KEY,
        num_prototypes=NUM_PROTOTYPES,
        load_umap=True,
        skip_eval=True,
    )

    with torch.no_grad():
        z = t.encode_adata(t.train_ds.adata, t.model, z_idx=1)
        scores = t.model.prototypes(z)
        S = F.softmax(scores / t.epsilon, dim=1).cpu().numpy()
    cell_ids = t.train_ds.adata.obs_names.to_numpy()

    S_topk = _topk_sparsify_rows(S, k=20)
    save_soft_assignments(run_dir, S_topk, cell_ids)
    print(f"scProto ({AFFINITY_TYPE}, pum={pum}) soft assignments saved to {run_dir}")

scProto (bk08, pum=ema) soft assignments already saved at /content/drive/MyDrive/models/fibnsc/proto_umap_ds-fibn_NP32_prtInit-wayp_aff-bk08_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31 -- skipping.


## Step 3d — scProto-only DGE report, all `PROTO_USAGE_MODES` x hard/soft (no SEACells needed)

All prerequisites for this are already done at this point (Step 2 -> hard `cell_assignments.csv`
per `pum`, Step 3c -> soft `soft_assignments.npz` per `pum`) -- none of it needs Steps 3/3b
(SEACells) to have run, so this gives scProto's own numbers immediately.

Reports Branch 1/2 for all 2×2 = 4 combinations (`'ema'`/`'nk'` × hard/soft) side by side,
plus `size_concentration_summary` (gini, effective_n_metacells, top5_share_of_cells) for all
four -- the direct answer to two questions at once: does `'nk'` actually prevent the
collapse `'ema'` didn't (compare HARD rows across `pum`), and does soft-weighting rescue
either mode's DGE numbers without fixing the underlying collapse (compare HARD vs SOFT within
each `pum` — if soft's `effective_n_metacells` stays close to hard's, the collapse is baked
into the learned distribution itself, not just the argmax step).

In [ ]:
# scProto-only DGE report -- all PROTO_USAGE_MODES x hard/soft, neither needs
# SEACells (Steps 3/3b) to have run.
GT_PATH = os.path.join(CODE_DIR, 'files', f'celltype_niches_full_{DS_ID}.csv')
if os.path.exists(GT_PATH):
    print(f'Loading cached ground truth from {GT_PATH}')
    ground_truth = load_ground_truth(GT_PATH)
else:
    ground_truth = compute_ground_truth(adata, CT_KEY, NICHE_KEY, min_pos=5, min_ctrl=20)
    save_ground_truth(ground_truth, GT_PATH)
    print(f'Saved ground truth ({len(ground_truth)} niche pairs) to {GT_PATH}')
print(f'{len(ground_truth)} niche pairs for {CT_VALUE}')

scproto_mc_labels, scproto_pseudobulk = {}, {}
scproto_branch1, scproto_branch2 = {}, {}
scproto_reports, scproto_sizes = {}, {}

for pum in PROTO_USAGE_MODES:
    run_dir = scproto_run_dirs[pum]

    # --- hard ---
    name_h = f'scProto ({pum}) HARD'
    ca = load_cell_assignments(run_dir, adata, NICHE_KEY)
    ml = majority_label_metacells(ca, CT_KEY, NICHE_KEY)
    pb = build_pseudobulk(ca, adata)
    b1 = branch1_recovery(pb, ml, ground_truth)
    b2 = branch2_recovery(pb, ml, ground_truth)
    scproto_mc_labels[name_h], scproto_pseudobulk[name_h] = ml, pb
    scproto_branch1[name_h], scproto_branch2[name_h] = b1, b2
    scproto_reports[name_h] = pd.concat([macro_average(b1, ['pearson_r', 'kendall_tau']),
                                          macro_average(b2, ['tpr'])])
    scproto_sizes[name_h] = size_concentration_summary(ml['n_cells'], name=name_h)

    # --- soft ---
    name_s = f'scProto ({pum}) SOFT'
    S, cell_ids = load_soft_assignments(run_dir)
    ml_s = soft_label_metacells(S, cell_ids, adata, CT_KEY, NICHE_KEY)
    pb_s = build_soft_pseudobulk(S, cell_ids, adata)
    b1_s = branch1_recovery(pb_s, ml_s, ground_truth)
    b2_s = branch2_recovery(pb_s, ml_s, ground_truth)
    scproto_mc_labels[name_s], scproto_pseudobulk[name_s] = ml_s, pb_s
    scproto_branch1[name_s], scproto_branch2[name_s] = b1_s, b2_s
    scproto_reports[name_s] = pd.concat([macro_average(b1_s, ['pearson_r', 'kendall_tau']),
                                          macro_average(b2_s, ['tpr'])])
    scproto_sizes[name_s] = size_concentration_summary(ml_s['n_cells'], name=name_s)

print(f"\nscProto -- {' vs. '.join(PROTO_USAGE_MODES)}, HARD vs. SOFT-weighted DGE recovery:")
display(pd.DataFrame(scproto_reports).T.round(3))

print(f"\nscProto -- metacell size concentration, all variants:")
display(pd.DataFrame(list(scproto_sizes.values())).set_index('run').round(3))

Loading cached ground truth from /content/drive/MyDrive/codes/interpretable-prototype/files/celltype_niches_full_fibnsc.csv
9 niche pairs for Fibroblasts

scProto -- ema, HARD vs. SOFT-weighted DGE recovery:


,pearson_r,kendall_tau,tpr
scProto (ema) HARD,0.631,0.439,0.962
scProto (ema) SOFT,0.890,0.692,0.986



scProto -- metacell size concentration, all variants:


,n_metacells_used,median_size,gini,effective_n_metacells,top5_share_of_cells
run,,,,,
scProto (ema) HARD,32,6.000,0.778,7.659,0.718
scProto (ema) SOFT,32,53.289,0.698,9.661,0.633


### Step 3d full detail — every niche, every metric (not just the two aggregate rows above)

Same `per_pair_diagnostics` breakdown Steps 4/5 use, applied to the HARD and SOFT results
already computed above (`mc_labels_h`/`b1_h`/`b2_h` and `mc_labels_s`/`b1_s`/`b2_s`) — no
SEACells dependency, works standalone off Step 2 + Step 3c alone.

In [ ]:
scproto_only_diag = {
    name: per_pair_diagnostics(scproto_mc_labels[name], scproto_branch1[name], scproto_branch2[name])
    for name in scproto_mc_labels
}
scproto_only_diag_long = pd.concat(
    [d.assign(method=name) for name, d in scproto_only_diag.items()], ignore_index=True,
)

metric_cols = ['pearson_r', 'kendall_tau', 'tpr', 'median_pos_mc_size', 'mean_niche_purity']
scproto_only_wide = {
    m: scproto_only_diag_long.pivot(index='niche', columns='method', values=m)
    for m in metric_cols
}

for m in metric_cols:
    print(f"--- {m} by niche, all scProto variants ---")
    print(scproto_only_wide[m].round(3).to_string())
    print()

print("--- full long-format table: every (niche, variant) row, every column ---")
full_cols = ['niche', 'method', 'n_pos_mc', 'median_pos_mc_size', 'min_pos_mc_size',
             'mean_niche_purity', 'n_genes', 'n_ctrl_mc', 'pearson_r', 'kendall_tau', 'tpr']
full_cols = [c for c in dict.fromkeys(full_cols) if c in scproto_only_diag_long.columns]
print(scproto_only_diag_long[full_cols].round(3).to_string(index=False))

--- pearson_r by niche, all scProto variants ---
method                    scProto (ema) HARD  scProto (ema) SOFT
niche                                                           
Airways                                  NaN                 NaN
Alveolar spaces                        0.654                 NaN
Desmoplastic stroma                    0.500               0.922
Excluded                                 NaN                 NaN
Macrophage islands                       NaN                 NaN
Smooth muscle structures                 NaN                 NaN
T cell aggregates                      0.671               0.856
Tumor surface                          0.799               0.962
Vascular stroma                        0.530               0.820

--- kendall_tau by niche, all scProto variants ---
method                    scProto (ema) HARD  scProto (ema) SOFT
niche                                                           
Airways                                  NaN          

## Step 4 — redo the niche transcriptional-program recovery evaluation

Same pipeline as `plan1_niche_recovery_eval.ipynb` (Setup A/B/C + Branch 1/2), imported from
`interpretable_ssl.evaluation.niche_program_recovery` rather than redefined here. Ground truth
is naturally scoped to Fibroblasts only now (`compute_ground_truth` loops over
`adata.obs[CT_KEY].unique()`, which is a single value in this subset).

Now a five-way comparison, **all at the same `NUM_PROTOTYPES`**: scProto (`AFFINITY_TYPE`,
`'ema'`), scProto (`AFFINITY_TYPE`, `'nk'`), SEACells (PCA-native, Step 3), SEACells (`arbf`,
Step 3b), SEACells (`AFFINITY_TYPE`, Step 3b). `adata` was already loaded once in Step 1.5 and
is reused here — no second `sc.read_h5ad` of the same file.

In [ ]:
# adata already loaded in Step 1.5 -- reused here, not reloaded.
print(adata)

print(adata.obs[NICHE_KEY].value_counts(dropna=False))

AnnData object with n_obs × n_vars = 15309 × 960
    obs: 'section', 'celltypes', 'niches_3D', 'niches_2D', 'fibroblast_subclusters', 'tumor_pseudotime_rank', 'EMT_niche'
    uns: 'EMT_niche_colors', 'celltypes_colors', 'fibroblast_subclusters_colors', 'log1p', 'niches_2D_colors', 'niches_3D_colors', 'pca', 'spatial_subset_calibrated_radius', 'spatial_subset_ct_key', 'spatial_subset_ct_value', 'spatial_subset_src_ds_id', 'spatial_subset_target_median_neighbours'
    obsm: 'X_bk08', 'X_covet', 'X_ctx', 'X_pca', 'X_umap_2D_neighbourhoods', 'X_umap_3D_neighbourhoods', 'X_umap_SCT', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities'
    layers: 'SCT', 'counts', None (.X)
niches_2D
Desmoplastic stroma         5856
Tumor surface               3147
Vascular stroma             2179
Macrophage islands          1047
T cell aggregates           1001
Excluded                     525
Tumor core                   494
Alveolar spaces              415
Airways                      363
Smooth muscle s

In [ ]:
GT_PATH = os.path.join(CODE_DIR, 'files', f'celltype_niches_full_{DS_ID}.csv')

if os.path.exists(GT_PATH):
    print(f'Loading cached ground truth from {GT_PATH}')
    ground_truth = load_ground_truth(GT_PATH)
else:
    ground_truth = compute_ground_truth(adata, CT_KEY, NICHE_KEY, min_pos=5, min_ctrl=20)
    save_ground_truth(ground_truth, GT_PATH)
    print(f'Saved ground truth ({len(ground_truth)} niche pairs) to {GT_PATH}')

print(f'{len(ground_truth)} niche pairs for {CT_VALUE}')

Loading cached ground truth from /content/drive/MyDrive/codes/interpretable-prototype/files/celltype_niches_full_fibnsc.csv
9 niche pairs for Fibroblasts


In [ ]:
run_dirs = {f'scProto ({pum})': scproto_run_dirs[pum] for pum in PROTO_USAGE_MODES}
run_dirs['SEACells (PCA)'] = get_seacell_model_dir(DS_ID, num_prototypes=NUM_PROTOTYPES)
for graph_name in dict.fromkeys(['arbf', AFFINITY_TYPE]):
    run_dirs[f'SEACells ({graph_name})'] = seacell_aff_run_dirs[graph_name]

missing = {
    name: d for name, d in run_dirs.items()
    if not d or not os.path.exists(os.path.join(d, 'cell_assignments.csv'))
}
assert not missing, f'missing cell_assignments.csv for: {missing} -- run Steps 2/3/3b first'
assert len(set(run_dirs.values())) == len(run_dirs), (
    f'two methods resolved to the SAME run dir -- this was a real bug before '
    f'resolve_scproto_run_dir/exact-path SEACells lookups fixed it: {run_dirs}'
)
run_dirs

{'scProto (ema)': '/content/drive/MyDrive/models/fibnsc/proto_umap_ds-fibn_NP32_prtInit-wayp_aff-bk08_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
 'SEACells (PCA)': '/content/drive/MyDrive/models/fibnsc/seacell_K32',
 'SEACells (arbf)': '/content/drive/MyDrive/models/fibnsc/seacell_arbf',
 'SEACells (bk08)': '/content/drive/MyDrive/models/fibnsc/seacell_bk08'}

In [ ]:
cell_assign = {
    name: load_cell_assignments(d, adata, NICHE_KEY) for name, d in run_dirs.items()
}
mc_labels = {
    name: majority_label_metacells(df, CT_KEY, NICHE_KEY) for name, df in cell_assign.items()
}
pseudobulk = {
    name: build_pseudobulk(df, adata) for name, df in cell_assign.items()
}

for name, df in mc_labels.items():
    print(f"{name}: n_metacells={len(df)}  median_size={df['n_cells'].median():.0f}  "
          f"min_size={df['n_cells'].min()}  max_size={df['n_cells'].max()}  "
          f"frac_below_10={(df['n_cells'] < 10).mean():.1%}")

scProto (ema): n_metacells=32  median_size=6  min_size=4  max_size=2756  frac_below_10=65.6%
SEACells (PCA): n_metacells=32  median_size=425  min_size=119  max_size=1046  frac_below_10=0.0%
SEACells (arbf): n_metacells=32  median_size=406  min_size=101  max_size=1864  frac_below_10=0.0%
SEACells (bk08): n_metacells=32  median_size=462  min_size=104  max_size=1230  frac_below_10=0.0%


In [ ]:
branch1 = {
    name: branch1_recovery(pseudobulk[name], mc_labels[name], ground_truth)
    for name in run_dirs
}
branch2 = {
    name: branch2_recovery(pseudobulk[name], mc_labels[name], ground_truth)
    for name in run_dirs
}

report = pd.DataFrame({
    name: pd.concat([
        macro_average(branch1[name], ['pearson_r', 'kendall_tau']),
        macro_average(branch2[name], ['tpr']),
    ]) for name in run_dirs
}).T
report.index.name = 'method'
print(report.round(3).to_string())

                 pearson_r  kendall_tau    tpr
method                                        
scProto (ema)        0.631        0.439  0.962
SEACells (PCA)       0.852        0.605  0.990
SEACells (arbf)      0.889        0.654  0.975
SEACells (bk08)      0.916        0.706  0.998


## Diagnostics — full per-niche breakdown (every metric, not just the aggregate)

The `report` above averages over methods and niches into one number each; with only 9 niches
for Fibroblasts, that hides more than it shows. This cell prints **every** metric
(`pearson_r`, `kendall_tau`, `tpr`, metacell size, niche purity) for **every** (niche, method)
pair, both as per-metric wide pivots (niche x method, easy to scan one metric at a time) and
as the full long-format table (one row per niche x method, every column at once, nothing
aggregated away) — so a gap or a tie can be checked against the underlying numbers directly
instead of trusting a single averaged figure.

In [ ]:
diag = {
    name: per_pair_diagnostics(mc_labels[name], branch1[name], branch2[name])
    for name in run_dirs
}
diag_long = pd.concat(
    [d.assign(method=name) for name, d in diag.items()], ignore_index=True,
)

metric_cols = ['pearson_r', 'kendall_tau', 'tpr', 'median_pos_mc_size', 'mean_niche_purity']
wide = {m: diag_long.pivot(index='niche', columns='method', values=m) for m in metric_cols}

scproto_cols = [c for c in wide['pearson_r'].columns if c.startswith('scProto')]
seacells_cols = [c for c in wide['pearson_r'].columns if c.startswith('SEACells')]
wide['pearson_r']['best_scproto'] = wide['pearson_r'][scproto_cols].max(axis=1)
wide['pearson_r']['best_seacells'] = wide['pearson_r'][seacells_cols].max(axis=1)
wide['pearson_r']['scproto_gap'] = wide['pearson_r']['best_seacells'] - wide['pearson_r']['best_scproto']
niche_order = wide['pearson_r'].sort_values('scproto_gap', ascending=False).index

n_wins = int((wide['pearson_r']['scproto_gap'] < 0).sum())
n_total = int(wide['pearson_r']['best_scproto'].notna().sum())
print(f"best scProto variant ({scproto_cols}) beats the best SEACells variant "
      f"({seacells_cols}) on pearson_r for {n_wins}/{n_total} niches\n")

for m in metric_cols:
    print(f"--- {m} by niche x method "
          f"{'(sorted by best_seacells - best_scproto gap, worst-for-scProto first)' if m == 'pearson_r' else ''} ---")
    print(wide[m].loc[niche_order].round(3).to_string())
    print()

print("--- full long-format table: every (niche, method) row, every column ---")
full_cols = ['niche', 'method', 'n_pos_mc', 'median_pos_mc_size', 'min_pos_mc_size',
             'mean_niche_purity', 'n_genes', 'n_pos_mc', 'n_ctrl_mc',
             'pearson_r', 'kendall_tau', 'tpr']
full_cols = [c for c in dict.fromkeys(full_cols) if c in diag_long.columns]  # dedupe, keep order
print(diag_long.set_index('niche').loc[niche_order].reset_index()[full_cols]
      .round(3).to_string(index=False))

best scProto variant (['scProto (ema)']) beats the best SEACells variant (['SEACells (PCA)', 'SEACells (arbf)', 'SEACells (bk08)']) on pearson_r for 0/5 niches

--- pearson_r by niche x method (sorted by best_seacells - best_scproto gap, worst-for-scProto first) ---
method                    SEACells (PCA)  SEACells (arbf)  SEACells (bk08)  scProto (ema)  best_scproto  best_seacells  scproto_gap
niche                                                                                                                              
Vascular stroma                      NaN              NaN            0.909          0.530         0.530          0.909        0.378
Desmoplastic stroma                0.741            0.830            0.813          0.500         0.500          0.830        0.330
T cell aggregates                    NaN            0.883            0.928          0.671         0.671          0.928        0.257
Tumor surface                      0.964            0.955            0.97

## Step 5 — soft-label-based metacell labeling + DGE recovery

Same Setup B/C + Branch 1/2 pipeline as Step 4, but a metacell's label/purity/size and its
pseudobulk are built from the **soft** assignment matrix (`soft_label_metacells`,
`build_soft_pseudobulk` — `niche_program_recovery.py`) instead of a hard groupby over
`cell_assignments.csv`: every cell contributes to every metacell in proportion to its
assignment weight, not just the one it hard-argmaxes into. `branch1_recovery`/
`branch2_recovery`/`macro_average`/`per_pair_diagnostics` are reused completely unchanged —
they only ever consumed `(pseudobulk_df, mc_labels_df)`, and don't care whether those came
from a hard groupby or a soft weighted sum.

Every method with a saved soft matrix is eligible: both scProto variants (Step 3c) and both
`run_seacells_on_latent`-based SEACells variants (`arbf` and `AFFINITY_TYPE`, Step 3b — that
function always saves one). SEACells (PCA-native, Step 3, plain `train_seacell`) has no soft
matrix and is left out of this section.

This directly tests whether scProto's collapse is a **labeling-time artifact of forcing a
hard argmax on an otherwise well-calibrated soft distribution**, or a **deeper problem with
the soft distribution itself** (e.g. most of its probability mass already concentrated on
very few prototypes, in which case soft-weighting won't rescue it either) — for both
`PROTO_USAGE_MODES`. Reports the same full per-niche breakdown as Step 4's diagnostics —
every eligible method's hard and soft numbers side by side, not just one aggregate number.

In [ ]:
soft_run_dirs = {}
for pum in PROTO_USAGE_MODES:
    soft_run_dirs[f'scProto ({pum}) [soft]'] = run_dirs[f'scProto ({pum})']
for graph_name in dict.fromkeys(['arbf', AFFINITY_TYPE]):
    soft_run_dirs[f'SEACells ({graph_name}) [soft]'] = run_dirs[f'SEACells ({graph_name})']

missing_soft = {
    name: d for name, d in soft_run_dirs.items()
    if not os.path.exists(os.path.join(d, 'soft_assignments.npz'))
}
assert not missing_soft, f'missing soft_assignments.npz for: {missing_soft} -- run Step 3b/3c first'

soft_mats = {name: load_soft_assignments(d) for name, d in soft_run_dirs.items()}
soft_mc_labels = {
    name: soft_label_metacells(S, cell_ids, adata, CT_KEY, NICHE_KEY)
    for name, (S, cell_ids) in soft_mats.items()
}
soft_pseudobulk = {
    name: build_soft_pseudobulk(S, cell_ids, adata)
    for name, (S, cell_ids) in soft_mats.items()
}

for name, df in soft_mc_labels.items():
    print(f"{name}: n_metacells={len(df)}  median_eff_size={df['n_cells'].median():.1f}  "
          f"min_eff_size={df['n_cells'].min():.1f}  max_eff_size={df['n_cells'].max():.1f}  "
          f"frac_below_10={(df['n_cells'] < 10).mean():.1%}")

soft_branch1 = {
    name: branch1_recovery(soft_pseudobulk[name], soft_mc_labels[name], ground_truth)
    for name in soft_run_dirs
}
soft_branch2 = {
    name: branch2_recovery(soft_pseudobulk[name], soft_mc_labels[name], ground_truth)
    for name in soft_run_dirs
}

soft_report = pd.DataFrame({
    name: pd.concat([
        macro_average(soft_branch1[name], ['pearson_r', 'kendall_tau']),
        macro_average(soft_branch2[name], ['tpr']),
    ]) for name in soft_run_dirs
}).T
soft_report.index.name = 'method'

print()
print("--- hard (Step 4) vs. soft (this step) report, side by side ---")
print(pd.concat([report, soft_report]).round(3).to_string())

scProto (ema) [soft]: n_metacells=32  median_eff_size=53.3  min_eff_size=26.2  max_eff_size=2246.8  frac_below_10=0.0%
SEACells (arbf) [soft]: n_metacells=32  median_eff_size=445.7  min_eff_size=122.1  max_eff_size=1385.7  frac_below_10=0.0%
SEACells (bk08) [soft]: n_metacells=32  median_eff_size=469.0  min_eff_size=232.5  max_eff_size=829.7  frac_below_10=0.0%

--- hard (Step 4) vs. soft (this step) report, side by side ---
                        pearson_r  kendall_tau    tpr
method                                               
scProto (ema)               0.631        0.439  0.962
SEACells (PCA)              0.852        0.605  0.990
SEACells (arbf)             0.889        0.654  0.975
SEACells (bk08)             0.916        0.706  0.998
scProto (ema) [soft]        0.890        0.692  0.986
SEACells (arbf) [soft]      0.797        0.555  0.978
SEACells (bk08) [soft]      0.940        0.758  1.000


In [ ]:
soft_diag = {
    name: per_pair_diagnostics(soft_mc_labels[name], soft_branch1[name], soft_branch2[name])
    for name in soft_run_dirs
}
soft_diag_long = pd.concat(
    [d.assign(method=name) for name, d in soft_diag.items()], ignore_index=True,
)

metric_cols = ['pearson_r', 'kendall_tau', 'tpr', 'median_pos_mc_size', 'mean_niche_purity']
soft_wide = {m: soft_diag_long.pivot(index='niche', columns='method', values=m) for m in metric_cols}

for m in metric_cols:
    print(f"--- soft {m} by niche x method ---")
    print(soft_wide[m].round(3).to_string())
    print()

print("--- full soft long-format table: every (niche, method) row, every column ---")
full_cols = ['niche', 'method', 'n_pos_mc', 'median_pos_mc_size', 'min_pos_mc_size',
             'mean_niche_purity', 'pearson_r', 'kendall_tau', 'tpr']
full_cols = [c for c in dict.fromkeys(full_cols) if c in soft_diag_long.columns]
print(soft_diag_long[full_cols].round(3).to_string(index=False))

--- soft pearson_r by niche x method ---
method                    SEACells (arbf) [soft]  SEACells (bk08) [soft]  scProto (ema) [soft]
niche                                                                                         
Airways                                      NaN                   0.928                   NaN
Alveolar spaces                              NaN                     NaN                   NaN
Desmoplastic stroma                        0.633                   0.901                 0.922
Smooth muscle structures                     NaN                     NaN                   NaN
T cell aggregates                            NaN                     NaN                 0.856
Tumor core                                   NaN                     NaN                   NaN
Tumor surface                              0.961                   0.973                 0.962
Vascular stroma                              NaN                   0.958                 0.820

--- soft

## Summary

Compare `report` above against the whole-tissue scProto run from `plan1_niche_recovery_eval.ipynb`
(Pearson r=0.291, Kendall tau=0.205, TPR=0.545) and the `mc_labels` size distribution against
that run's (median 1-4 cells almost everywhere).

This notebook now separates three previously-confounded questions:
1. **Graph vs. clusterer**: compare SEACells (`arbf`) vs. SEACells (`AFFINITY_TYPE`) — same
   clusterer, different graph — against SEACells (`AFFINITY_TYPE`) vs. scProto
   (`AFFINITY_TYPE`) — same graph, different clusterer.
2. **`'ema'` vs. `'nk'` anti-collapse loss**: does `'nk'` (total soft-assignment mass per
   prototype) prevent the collapse `'ema'` (single most-confident cell) didn't? Compare
   `size_concentration_summary`/`frac_below_10` for scProto (`'ema'`) vs. scProto (`'nk'`) in
   Step 3d/4.
3. **Hard vs. soft labeling**: does soft-weighted labeling recover DGE metrics without
   actually fixing the collapse (`effective_n_metacells` staying low regardless), or does it
   genuinely rescue the result? Step 3d/5.
4. **Is `NUM_PROTOTYPES` itself the problem**: Step 1.5's Leiden resolution sweep checks
   whether the graph can even support as many real communities as requested, independent of
   which `pum` or hard/soft choice is made.

**Caveat**: this run's Fibroblasts come from `s28nsc` (the full section-28 tissue), while the
`plan1_niche_recovery_eval.ipynb` comparison numbers above come from `ss28nsc` (a subsample of
the same section). Same tissue/section, so biologically comparable, but not an identical cell
set — keep that in mind if the comparison is reported anywhere more precise than "the gap
narrowed/didn't narrow."
